In [ ]:
horizons = {
    "4h":  8, "8h": 16, "24h": 48,
    "7d":  336, "14d": 672, "1m": 1488,
}

In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 14.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
house = "house_1"

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" if h != "1m" else "Горизонт 1m (недоступно)"
                    for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (hn, hp) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    if hn not in cv_dfs:
        # пустой subplot для 1m
        fig.add_annotation(
            text="Недоступно для TFT(ограничение памяти GPU)",
            x=0.5, y=0.5,
            xref=f"x{i+1} domain", yref=f"y{i+1} domain",
            showarrow=False,
            font=dict(size=10, color="gray"),
            row=row, col=col
        )
        continue

    house_cv = cv_dfs[hn]
    cutoffs = house_cv["cutoff"].unique()

    n_windows = max(1, hp // 48)
    selected_cutoffs = cutoffs[::48][:n_windows]

    subsets = []
    for c in selected_cutoffs:
        tmp = house_cv[house_cv["cutoff"] == c]
        subsets.append(tmp)

    subset = pd.concat(subsets).drop_duplicates(subset="ds").sort_values("ds")

    fig.add_trace(go.Scatter(
        x=subset["ds"], y=subset["y"],
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=subset["ds"], y=subset["TFT"],
        mode="lines", name="прогноз TFT",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="TFT: Прогнозные и фактические значения по всем горизонтам (дом 1)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image("35_tft_forecast_all_horizons.png", scale=2)